In [1]:
import os
import sys
os.chdir('..')
current_dir = os.path.abspath('')
sys.path.append(current_dir)

In [2]:
import scripts.helpers.helpers
import scripts.helpers.base_helpers
from pytorch_lightning import seed_everything
import importlib
from tqdm import tqdm
import pickle

In [3]:
importlib.reload(scripts.helpers.helpers)
importlib.reload(scripts.helpers.base_helpers)
from scripts.helpers.helpers import *
from scripts.helpers.base_helpers import *

In [4]:
seed_everything(42)
SAVE_PATH = "outputs/text_parsed_video_diffusion/"
set_lowvram_mode(False)
model = load_model_from_joblib("./checkpoints/sdxlbase1_cache.joblib")

Global seed set to 42


In [5]:
options = {
  "discretization": "LegacyDDPMDiscretization",
  "sigma_min"     : 0.03,   #EDMDiscretization, [-inf, inf | 0.03]
  "sigma_max"     : 14.61,  #EDMDiscretization, [-inf, inf | 14.61]
  "rho"           : 3.0,    #EDMDiscretization, [-inf, inf | 3.0]

  "guider"                  : "VanillaCFG",
  "additional_guider_kwargs": {},
  "vanilla_cfg"             : 2.0,  #VanillaCFG, [0.0, inf | 5.0]
  "linear_cfg"              : 1.5,  #LinearCFG, [1.0, inf | 1.5]
  "triangle_cfg"            : 2.5,  #TriangleCFG, [1.0, 10.0 | 2.5]
  "min_cfg"                 : 1.0,  #LinearCFG TriangleCFG, [1.0]
  "num_frames"              : 25,   #LinearCFG TriangleCFG, [25]

  "sampler"           : "HeunEDMSampler",
  "s_churn"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "s_tmin"            : 0.0,    #EulerEDM HeunEDM [0.0, inf |0.0]
  "s_tmax"            : 999.0,  #EulerEDM HeunEDM [0.0, inf | 999.0]
  "s_noise"           : 0.0,    #EulerEDM HeunEDM [0.0, inf | 0.0]
  "eta"               : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "s_noise_ancestral" : 1.0,    #EulerAncestral DPMPP2SAncestral [0.0, inf | 1.0]
  "order"             : 4,      #LinearMultistep [1, inf | 4]

  "crop_coords_top"         : 0,    #[0, inf | 0]
  "crop_coords_left"        : 0,    #[0, inf | 0]
  "aesthetic_score"         : 6.0,  #[-inf, inf | 6.0]
  "negative_aesthetic_score": 2.5,  #[-inf, inf | 2.5]
  "fps"                     : 6,    #[1, inf | 6]
  "mb_id"                   : 127,  #[0, 511 | 127]
  "image_path"              : None
}

In [57]:
seq = {
"0": "A man walking a dog beside a serene lake at sunset, golden light reflecting off the water, lush trees surrounding the area, photorealistic style, high detail",
"1": "Man and dog strolling along lakeside path, evening sky with vibrant orange and purple hues, calm water surface, photorealistic style, high detail",
"2": "Peaceful lake scene with man and dog, sudden bright light appearing in sky above, mysterious glow illuminating surroundings, photorealistic style, high detail",
"3": "Alien spaceship hovering over lake, tractor beam emanating downwards, man and dog looking up in awe, trees swaying from force, photorealistic style, high detail",
"4": "Man being lifted off ground by alien tractor beam, dog barking frantically below, spaceship's metallic surface reflecting evening light, photorealistic style, high detail",
"5": "Man ascending towards alien spaceship, lake and landscape shrinking below, dog becoming a tiny speck, photorealistic style, high detail",
"6": "Interior of alien spaceship, man floating in strange anti-gravity chamber, bizarre alien technology surrounding him, otherworldly lighting, photorealistic style, high detail",
"7": "Kaleidoscopic tunnel of light and energy, man's body stretching and distorting as he travels through space-time, photorealistic style, high detail",
"8": "Indescribable alien world, impossible geometric shapes floating in void, colors beyond human perception, man's figure barely visible, photorealistic style, high detail",
"9": "Fractal landscape of ever-changing patterns, man's consciousness expanding beyond his body, cosmic energy swirling around, photorealistic style, high detail",
"10": "Non-Euclidean architecture of an alien city, multiple dimensions intersecting, man's perception fragmenting into countless parallel realities, photorealistic style, high detail",
"11": "Timeless void where past, present, and future coexist, echoes of man's memories and potential futures visible as translucent images, photorealistic style, high detail",
"12": "Cosmic web of interconnected consciousnesses, man's mind linking with vast alien intelligence, energy tendrils connecting all things, photorealistic style, high detail",
"13": "Swirling vortex of reality collapsing, man's figure reassembling from scattered particles, boundaries of space-time breaking down, photorealistic style, high detail",
"14": "Blinding flash of light engulfing everything, man's silhouette barely visible in the center, reality resetting, photorealistic style, high detail",
"15": "Man reappearing above lake in beam of light, alien spaceship vanishing into evening sky, ripples spreading across water surface, photorealistic style, high detail",
"16": "Man gently descending towards lake shore, disoriented expression, dog running towards him, trees still swaying from spaceship's departure, photorealistic style, high detail",
"17": "Man landing softly on lakeside path, looking around in confusion, dog jumping up to greet him, evening sky returning to normal, photorealistic style, high detail",
"18": "Lakeside scene almost back to normal, man kneeling to hug his dog, faint glow in sky where spaceship disappeared, photorealistic style, high detail",
"19": "Serene lake at evening, man and dog resuming their walk, everything appears unchanged except for man's bewildered expression, photorealistic style, high detail"
}

In [77]:
prompts = list(seq.values())
num_steps =  20
dims = [576, 1024] #height, width
sampler = init_sampling(options = options, steps = num_steps)

In [78]:
c0, uc0 = get_conditionings(model, dims, [prompts[0]])
z0 = get_samples(model, sampler, dims, c0, uc0)

##############################  Sampling setting  ##############################
Sampler: HeunEDMSampler
Discretization: LegacyDDPMDiscretization
Guider: VanillaCFG


Sampling with HeunEDMSampler for 20 steps: 100%|██████████| 20/20 [00:03<00:00,  5.03it/s]


In [79]:
sampler.discretization = Img2ImgDiscretizationWrapper(
    sampler.discretization, strength=0.95
)
sigmas = sampler.discretization(sampler.num_steps).cuda()
sigma = sigmas[0]

In [80]:
sigma

tensor(8.3028, device='cuda:0')

In [81]:
samples_z = [z0]
def denoiser(x, sigma, c):
    return model.denoiser(model.model, x, sigma, c)
verbose_save = sampler.verbose
sampler.verbose = False

for prompt in tqdm(prompts[1:], desc="Stepping through prompts"):
    ctp1, uctp1 = get_conditionings(model, dims, [prompt])
    zt = samples_z[-1]
    noise = torch.randn_like(zt)
    noised_z = zt + noise * sigma
    noised_z = noised_z / torch.sqrt(
        1.0 + sigmas[0] ** 2.0
    )
    with torch.no_grad():
        with autocast("cuda"):
            with model.ema_scope():
                ztp1 = sampler(denoiser, noised_z, cond=ctp1, uc=uctp1)
    samples_z.append(copy.deepcopy(ztp1))
    clear_vram()

samples_z = torch.cat(samples_z, dim=0)
sampler.verbose = verbose_save

Stepping through prompts: 100%|██████████| 19/19 [01:11<00:00,  3.76s/it]


In [84]:
samples_z_interp = interpolate_samples(samples_z, 18)

In [85]:
samples = decode_samples(model, samples_z_interp)

In [86]:
save_mp4(SAVE_PATH, samples, fps=12)

In [83]:
save_png(SAVE_PATH, samples)

In [11]:
clear_vram()